# C0.2 — Colab rate check + vLLM smoke (FinSight)

Run once per GPU. **T4 and A100: rate check only** (set `RUN_VLLM = False`). **L4: also the vLLM
test** (`RUN_VLLM = True`): `awq_marlin` needs compute capability 8.0+, so it cannot run on a T4.
Read the compute units in the Colab Resources panel before and after the run and type them in the
parameter cell. Keep the run about 10 minutes. Steps: `docs/phase3/COLAB_STEPS_rate_check.md`.

In [ ]:
# ---- parameters (edit, then Run all) ------------------------------------------------------
UNITS_BEFORE = None      # compute units shown in the Resources panel before the run
UNITS_AFTER = None       # ... and after the run (type it before running the last cell)
RUN_VLLM = False         # True on the L4 only
MODEL = "Qwen/Qwen3-4B-AWQ"   # small AWQ model for the smoke test
# candidate vLLM pins, tried in order; the first that imports and generates becomes THE pin
VLLM_CANDIDATES = ["vllm==0.11.0", "vllm==0.10.2", "vllm"]
MATMUL_SECONDS = 120
UNASSIGN = False         # True disconnects the runtime when the last cell ends

In [ ]:
# Shared helpers: copy notebooks/colab/_common.py to MyDrive/FinSight/_common.py once (COLAB_STEPS).
import sys, time
sys.path.insert(0, "/content/drive/MyDrive/FinSight")
from google.colab import drive
drive.mount("/content/drive")
from _common import mount_drive, Checkpointer, run_summary, gpu_info, unassign_runtime
STARTED = time.time()
JOB_DIR = mount_drive("rate_check")
print(gpu_info())

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > pick a GPU"
n, t0, iters = 4096, time.time(), 0
a = torch.randn(n, n, device="cuda", dtype=torch.float16); b = torch.randn(n, n, device="cuda", dtype=torch.float16)
while time.time() - t0 < MATMUL_SECONDS:
    for _ in range(50):
        a @ b
    torch.cuda.synchronize(); iters += 50
tflops = round(2 * n**3 * iters / (time.time() - t0) / 1e12, 2)
cap = torch.cuda.get_device_capability()
print({"fp16_tflops": tflops, "capability": cap, "torch": torch.__version__})
EXTRA = {"fp16_tflops": tflops, "capability": list(cap), "torch": torch.__version__}

In [ ]:
import subprocess, json
PIN_OK = None
if RUN_VLLM and cap[0] >= 8:
    for pin in VLLM_CANDIDATES:
        print("trying", pin)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=False)
        test = f"""
from vllm import LLM, SamplingParams
llm = LLM('{MODEL}', quantization='awq_marlin', max_model_len=2048, gpu_memory_utilization=0.85)
out = llm.generate(['Explain an IPO in one sentence.'], SamplingParams(max_tokens=40))
print('GEN_OK', out[0].outputs[0].text)
"""
        r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True)
        print(r.stdout[-800:], r.stderr[-800:])
        if "GEN_OK" in r.stdout:
            PIN_OK = pin
            break
    ver = subprocess.run([sys.executable, "-m", "pip", "show", "vllm"], capture_output=True, text=True).stdout
    EXTRA.update(vllm_pin=PIN_OK, vllm_pip_show=ver[:300], model=MODEL, quantization="awq_marlin")
else:
    print("vLLM test skipped (RUN_VLLM False or GPU below sm80)")

In [ ]:
s = run_summary(JOB_DIR, "rate_check", started=STARTED, units_before=UNITS_BEFORE,
                units_after=UNITS_AFTER, items_done=0, pins={"vllm": str(PIN_OK)}, extra=EXTRA)
print(json.dumps(s, indent=1))
print("Copy run_summary.json from Drive to eval_results/c/ and run scripts/log_compute.py on it.")
unassign_runtime(UNASSIGN)